# Phishing Email Detection Model Training

This notebook trains a machine learning model to detect phishing emails using the HuggingFace dataset.

**Dataset:** [zefang-liu/phishing-email-dataset](https://huggingface.co/datasets/zefang-liu/phishing-email-dataset)
- 18,650 email samples
- Labels: "Phishing Email" or "Safe Email"

**Model Architecture:**
- **Vectorizer:** TF-IDF (5000 features, unigrams + bigrams)
- **Classifier:** Logistic Regression (balanced class weights)

**Pipeline:**
1. Load dataset from HuggingFace
2. Preprocess text (clean, normalize - NO NLTK)
3. TF-IDF Vectorization
4. Train Logistic Regression Classifier
5. Evaluate and save model artifacts

In [ ]:
# Install required packages (run this first in Colab)
!pip install datasets scikit-learn joblib -q

In [1]:
# Imports - Simplified (NO NLTK required)
import pandas as pd
import numpy as np
import re
import json
import joblib
import os
from datetime import datetime, timezone
from datasets import load_dataset
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")
print("   No NLTK required - using simple regex preprocessing")

c:\Users\Zayad\Desktop\ACDS-FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful!
   No NLTK required - using simple regex preprocessing


In [2]:
# Load Dataset from HuggingFace
print("Loading dataset from HuggingFace...")
dataset = load_dataset("zefang-liu/phishing-email-dataset", split="train")

# Convert to DataFrame
df = pd.DataFrame(dataset)
print(f"Dataset loaded successfully!")
print(f"Total samples: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel Distribution:")
print(df['Email Type'].value_counts())
df.head()

Loading dataset from HuggingFace...


Generating train split: 100%|██████████| 18650/18650 [00:01<00:00, 11022.91 examples/s]



Dataset loaded successfully!
Total samples: 18,650
Columns: ['Unnamed: 0', 'Email Text', 'Email Type']

Label Distribution:
Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64


,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [3]:
# Text Preprocessing - Unified Pipeline (NO NLTK)
# This MUST match backend/ml/preprocess.py exactly

def preprocess_text(text):
    """
    Unified text preprocessing for email content.
    IMPORTANT: Keep in sync with backend/ml/preprocess.py
    """
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Replace URLs with token (preserve as indicator)
    text = re.sub(r'http[s]?://\S+|www\.\S+', ' URL_TOKEN ', text)
    
    # Replace email addresses with token
    text = re.sub(r'\S+@\S+\.\S+', ' EMAIL_TOKEN ', text)
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Keep alphanumeric and spaces only
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply preprocessing
print("Preprocessing text...")
df['clean_text'] = df['Email Text'].apply(preprocess_text)

# Create binary labels: 1 = Phishing, 0 = Safe
df['label'] = (df['Email Type'] == 'Phishing Email').astype(int)

print(f"✅ Preprocessing complete!")
print(f"\nSample original vs cleaned:")
print(f"Original: {df['Email Text'].iloc[0][:150]}...")
print(f"Cleaned:  {df['clean_text'].iloc[0][:150]}...")

Preprocessing text...
✅ Preprocessing complete!

Sample original vs cleaned:
Original: re : 6 . 1100 , disc : uniformitarianism , re : 1086 ; sex / lang dick hudson 's observations on us use of 's on ' but not 'd aughter ' as a vocative ...
Cleaned:  re 6 1100 disc uniformitarianism re 1086 sex lang dick hudson s observations on us use of s on but not d aughter as a vocative are very thought provok...
✅ Preprocessing complete!

Sample original vs cleaned:
Original: re : 6 . 1100 , disc : uniformitarianism , re : 1086 ; sex / lang dick hudson 's observations on us use of 's on ' but not 'd aughter ' as a vocative ...
Cleaned:  re 6 1100 disc uniformitarianism re 1086 sex lang dick hudson s observations on us use of s on but not d aughter as a vocative are very thought provok...


In [4]:
# Train-Test Split
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Maintain label distribution
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"\nTraining label distribution:")
print(y_train.value_counts())

Training set: 14,920 samples
Test set: 3,730 samples

Training label distribution:
label
0    9058
1    5862
Name: count, dtype: int64


In [5]:
# Create and Train the Model Pipeline - TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,          # Reduced for faster inference
        ngram_range=(1, 2),         # Unigrams and bigrams
        min_df=3,                   # Minimum document frequency
        max_df=0.9,                 # Maximum document frequency
        stop_words='english',
        lowercase=True              # Redundant but explicit
    )),
    ('clf', LogisticRegression(
        C=1.0,                      # Regularization strength
        max_iter=1000,              # Convergence iterations
        random_state=42,
        class_weight='balanced',   # Handle class imbalance
        solver='lbfgs',
        n_jobs=-1
    ))
])

print("Training TF-IDF + Logistic Regression model...")
pipeline.fit(X_train, y_train)
print("✅ Training complete!")

Training TF-IDF + Logistic Regression model...
✅ Training complete!
✅ Training complete!


In [6]:
# Model Evaluation
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print("=" * 50)
print("MODEL EVALUATION RESULTS")
print("=" * 50)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n📊 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"📈 ROC-AUC Score: {roc_auc:.4f}")

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Safe Email', 'Phishing Email']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("🔢 Confusion Matrix:")
print(f"   True Negatives (Safe→Safe): {cm[0][0]}")
print(f"   False Positives (Safe→Phishing): {cm[0][1]}")
print(f"   False Negatives (Phishing→Safe): {cm[1][0]}")
print(f"   True Positives (Phishing→Phishing): {cm[1][1]}")

MODEL EVALUATION RESULTS

📊 Accuracy: 0.9590 (95.90%)
📈 ROC-AUC Score: 0.9929

📋 Classification Report:
                precision    recall  f1-score   support

    Safe Email       0.98      0.95      0.97      2264
Phishing Email       0.93      0.97      0.95      1466

      accuracy                           0.96      3730
     macro avg       0.95      0.96      0.96      3730
  weighted avg       0.96      0.96      0.96      3730

🔢 Confusion Matrix:
   True Negatives (Safe→Safe): 2158
   False Positives (Safe→Phishing): 106
   False Negatives (Phishing→Safe): 47
   True Positives (Phishing→Phishing): 1419


In [7]:
# Cross-Validation Score
print("Running 5-fold cross-validation...")
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f"\n✅ Cross-Validation Scores: {cv_scores}")
print(f"✅ Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

Running 5-fold cross-validation...

✅ Cross-Validation Scores: [0.96280161 0.96782842 0.96380697 0.96414209 0.96983914]
✅ Mean CV Accuracy: 0.9657 (+/- 0.0054)

✅ Cross-Validation Scores: [0.96280161 0.96782842 0.96380697 0.96414209 0.96983914]
✅ Mean CV Accuracy: 0.9657 (+/- 0.0054)


In [8]:
# Test with Sample Emails
test_emails = [
    "URGENT: Your account has been compromised! Click here immediately to verify your identity or your account will be suspended.",
    "Hi team, please find attached the quarterly report. Let me know if you have any questions.",
    "Congratulations! You've won $1,000,000! Click now to claim your prize before it expires!",
    "Meeting reminder: Project sync at 3pm tomorrow in Conference Room B.",
    "Your PayPal account has unusual activity. Verify now: http://paypa1-secure.com/verify"
]

print("=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)

for i, email in enumerate(test_emails, 1):
    clean_email = preprocess_text(email)
    prediction = pipeline.predict([clean_email])[0]
    confidence = pipeline.predict_proba([clean_email])[0]
    
    label = "🚨 PHISHING" if prediction == 1 else "✅ SAFE"
    conf_score = confidence[1] if prediction == 1 else confidence[0]
    
    print(f"\n[Email {i}]")
    print(f"Text: {email[:80]}...")
    print(f"Prediction: {label} (Confidence: {conf_score*100:.1f}%)")

SAMPLE PREDICTIONS

[Email 1]
Text: URGENT: Your account has been compromised! Click here immediately to verify your...
Prediction: 🚨 PHISHING (Confidence: 98.0%)

[Email 2]
Text: Hi team, please find attached the quarterly report. Let me know if you have any ...
Prediction: ✅ SAFE (Confidence: 91.8%)

[Email 3]
Text: Congratulations! You've won $1,000,000! Click now to claim your prize before it ...
Prediction: 🚨 PHISHING (Confidence: 91.9%)

[Email 4]
Text: Meeting reminder: Project sync at 3pm tomorrow in Conference Room B....
Prediction: ✅ SAFE (Confidence: 94.3%)

[Email 5]
Text: Your PayPal account has unusual activity. Verify now: http://paypa1-secure.com/v...
Prediction: 🚨 PHISHING (Confidence: 51.3%)


In [9]:
# Save the Model and Metadata to backend/ml/models/
import os

# Path relative to notebook location (ml_training/) -> go up and into backend/ml/models/
output_dir = os.path.join(os.path.dirname(os.getcwd()), 'backend', 'ml', 'models')
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, 'phishing_model.pkl')
joblib.dump(pipeline, model_path)

# Calculate additional metrics for model info
precision = cm[1][1] / (cm[1][1] + cm[0][1]) if (cm[1][1] + cm[0][1]) > 0 else 0
recall = cm[1][1] / (cm[1][1] + cm[1][0]) if (cm[1][1] + cm[1][0]) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# Comprehensive model metadata
model_info = {
    "model_version": "2.0.0",
    "model_type": "TF-IDF + Logistic Regression",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "dataset": "zefang-liu/phishing-email-dataset",
    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc),
        "cv_mean_accuracy": float(cv_scores.mean()),
        "cv_std": float(cv_scores.std())
    },
    "confusion_matrix": {
        "true_negatives": int(cm[0][0]),
        "false_positives": int(cm[0][1]),
        "false_negatives": int(cm[1][0]),
        "true_positives": int(cm[1][1])
    },
    "training_info": {
        "training_samples": int(len(X_train)),
        "test_samples": int(len(X_test)),
        "features": 5000,
        "ngram_range": [1, 2]
    },
    "preprocessing": {
        "method": "regex-based",
        "url_token": "URL_TOKEN",
        "email_token": "EMAIL_TOKEN"
    }
}

model_info_path = os.path.join(output_dir, 'model_info.json')
with open(model_info_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"✅ Model saved to: {model_path}")
print(f"✅ Model info saved to: {model_info_path}")
print(f"\n📦 Model size: {os.path.getsize(model_path) / (1024*1024):.2f} MB")
print(f"\n📋 Model Info:")
print(json.dumps(model_info, indent=2))

✅ Model saved to: c:\Users\Zayad\Desktop\ACDS-FYP\backend\ml\models\phishing_model.pkl
✅ Model info saved to: c:\Users\Zayad\Desktop\ACDS-FYP\backend\ml\models\model_info.json

📦 Model size: 0.22 MB

📋 Model Info:
{
  "model_version": "2.0.0",
  "model_type": "TF-IDF + Logistic Regression",
  "trained_at": "2025-12-11T09:09:15.631964+00:00",
  "dataset": "zefang-liu/phishing-email-dataset",
  "metrics": {
    "accuracy": 0.9589812332439678,
    "precision": 0.9304918032786885,
    "recall": 0.9679399727148704,
    "f1_score": 0.9488465396188566,
    "roc_auc": 0.9928512116815063,
    "cv_mean_accuracy": 0.9656836461126005,
    "cv_std": 0.002685987253286052
  },
  "confusion_matrix": {
    "true_negatives": 2158,
    "false_positives": 106,
    "false_negatives": 47,
    "true_positives": 1419
  },
  "training_info": {
    "training_samples": 14920,
    "test_samples": 3730,
    "features": 5000,
    "ngram_range": [
      1,
      2
    ]
  },
  "preprocessing": {
    "method": "regex

In [ ]:
# Download model files from Colab
import os

# First, verify the files exist
print(" Checking if files exist...")

if os.path.exists('models/phishing_model.pkl'):
    size = os.path.getsize('models/phishing_model.pkl') / (1024*1024)
    print(f" phishing_model.pkl exists ({size:.2f} MB)")
else:
    print(" phishing_model.pkl NOT FOUND!")
    print("   Run the 'Save the Model' cell first!")

if os.path.exists('models/model_info.json'):
    print(" model_info.json exists")
else:
    print(" model_info.json NOT FOUND!")

# List all files in models folder
print("\n Contents of models/ folder:")
if os.path.exists('models'):
    for f in os.listdir('models'):
        print(f"   - {f}")
else:
    print("   models/ folder doesn't exist!")

# Download files
print("\n Starting download...")
try:
    from google.colab import files
    
    if os.path.exists('models/phishing_model.pkl'):
        files.download('models/phishing_model.pkl')
        print("phishing_model.pkl download triggered!")
    
    if os.path.exists('models/model_info.json'):
        files.download('models/model_info.json')
        print(" model_info.json download triggered!")
    
    print("\n Check your browser's Downloads folder!")
    print("   If download didn't start, check for popup blockers.")
    
except ImportError:
    print(" Not running in Google Colab!")
    print("   Files are saved locally at:", os.path.abspath('models/'))
except Exception as e:
    print(f" Error: {e}")

📁 Checking if files exist...
✅ phishing_model.pkl exists (20.50 MB)
✅ model_info.json exists

📂 Contents of models/ folder:
   - model_info.json
   - phishing_model.pkl

📥 Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ phishing_model.pkl download triggered!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ model_info.json download triggered!

🎉 Check your browser's Downloads folder!
   If download didn't start, check for popup blockers.


In [1]:
# Download using Colab's file browser (SIMPLEST METHOD)
import os

print("="*60)
print("📥 DOWNLOAD YOUR MODEL FILES")
print("="*60)

# Check files exist
pkl_exists = os.path.exists('models/phishing_model.pkl')
json_exists = os.path.exists('models/model_info.json')

if pkl_exists:
    pkl_size = os.path.getsize('models/phishing_model.pkl') / (1024*1024)
    print(f"\n✅ phishing_model.pkl exists ({pkl_size:.2f} MB)")
else:
    print("\n❌ phishing_model.pkl NOT FOUND - run Save Model cell first!")

if json_exists:
    json_size = os.path.getsize('models/model_info.json') / 1024
    print(f"✅ model_info.json exists ({json_size:.2f} KB)")
else:
    print("❌ model_info.json NOT FOUND - run Save Model cell first!")

if pkl_exists and json_exists:
    print("\n" + "="*60)
    print("📁 HOW TO DOWNLOAD:")
    print("="*60)
    print("""
1. Look at the LEFT SIDEBAR in Colab
2. Click the 📁 FOLDER ICON (Files)
3. Navigate to: models/
4. Right-click on 'phishing_model.pkl' → Download
5. Right-click on 'model_info.json' → Download
    """)
    
    # Also try the files.download method
    print("="*60)
    print("🔄 Attempting auto-download...")
    print("="*60)
    
    try:
        from google.colab import files
        print("\nDownloading phishing_model.pkl...")
        files.download('models/phishing_model.pkl')
        print("Downloading model_info.json...")
        files.download('models/model_info.json')
        print("\n✅ Check your browser downloads!")
    except Exception as e:
        print(f"\n⚠️ Auto-download failed: {e}")
        print("\n👆 Use the MANUAL METHOD above (sidebar → files → download)")
else:
    print("\n❌ Cannot download - files don't exist!")
    print("   Run cells 1-11 first, then come back here.")

📥 DOWNLOAD YOUR MODEL FILES

✅ phishing_model.pkl exists (20.50 MB)
✅ model_info.json exists (0.30 KB)

📁 HOW TO DOWNLOAD:

1. Look at the LEFT SIDEBAR in Colab
2. Click the 📁 FOLDER ICON (Files)
3. Navigate to: models/
4. Right-click on 'phishing_model.pkl' → Download
5. Right-click on 'model_info.json' → Download
    
🔄 Attempting auto-download...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Check your browser downloads!
